# Crafting an AI-Powered HR Assistant

This notebook builds a conversational chatbot that answers questions about Nestle's HR policy PDF using Azure OpenAI embeddings, Chroma DB, LangChain, and Gradio.

## 1. Install dependencies

Run the next cell once if the required libraries are not installed.

In [ ]:
%pip install -r requirements.txt

## 2. Import libraries and configure Azure OpenAI

In [ ]:
import os
from pathlib import Path

import chromadb
import gradio as gr
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import AzureOpenAI

load_dotenv()

required_vars = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_API_VERSION',
    'AZURE_OPENAI_CHAT_DEPLOYMENT',
    'AZURE_OPENAI_EMBEDDING_DEPLOYMENT',
]
missing_vars = [name for name in required_vars if not os.getenv(name)]
if missing_vars:
    raise ValueError(f'Missing Azure OpenAI settings: {", ".join(missing_vars)}')

BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / 'data' / '1728286846_the_nestle_hr_policy_pdf_2012.pdf'
CHROMA_DIR = BASE_DIR / 'chroma_db'
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION')
CHAT_DEPLOYMENT = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')
EMBEDDING_DEPLOYMENT = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')

client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT')
)

def embed_texts(texts):
    response = client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=texts)
    return [item.embedding for item in response.data]

def embed_query(text):
    response = client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=text)
    return response.data[0].embedding

PDF_PATH

## 3. Load the Nestle HR policy PDF

In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

len(documents), documents[0].metadata

## 4. Split the document into chunks for retrieval

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(documents)
len(chunks)

## 5. Create embeddings and store them in Chroma DB

In [ ]:
chunk_texts = [chunk.page_content for chunk in chunks]
chunk_metadatas = [chunk.metadata for chunk in chunks]
chunk_ids = [f'chunk-{index}' for index in range(len(chunks))]
chunk_embeddings = embed_texts(chunk_texts)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(name='nestle_hr_policy')
existing_ids = collection.get()['ids']
if existing_ids:
    collection.delete(ids=existing_ids)

collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    metadatas=chunk_metadatas,
    embeddings=chunk_embeddings,
)

collection.count()

## 6. Create the prompt template and question-answering function

In [ ]:
SYSTEM_PROMPT = '''You are an HR policy assistant for Nestle.
Answer the user's question using only the retrieved policy context.
If the answer is not available in the context, say that the document does not provide enough information.
Keep the answer clear, concise, and professional.
Always mention the source page numbers when possible.
'''

def retrieve_docs(question, k=4):
    results = collection.query(
        query_embeddings=[embed_query(question)],
        n_results=k,
    )

    docs = []
    for document, metadata in zip(results['documents'][0], results['metadatas'][0]):
        docs.append({'page_content': document, 'metadata': metadata})
    return docs

def format_context(retrieved_docs):
    parts = []
    for index, doc in enumerate(retrieved_docs, start=1):
        page = doc['metadata'].get('page', 'unknown')
        parts.append(f"Source {index} | page {page}\n{doc['page_content']}")
    return '\n\n'.join(parts)

def answer_question(question):
    retrieved_docs = retrieve_docs(question)
    context = format_context(retrieved_docs)
    response = client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': f'''Context:\n{context}\n\nQuestion:\n{question}''',
            },
        ],
    )
    return response.choices[0].message.content

def process_user_input(user_input):
    if not user_input or not user_input.strip():
        return 'Please enter a question about the HR policy.'
    return answer_question(user_input)


## 7. Build a Gradio chatbot interface

In [ ]:
import gradio as gr

def gradio_chat(message, history):
    history = history or []
    if not message or not message.strip():
        return '', history

    answer = process_user_input(message)
    history = history + [
        {'role': 'user', 'content': message},
        {'role': 'assistant', 'content': answer},
    ]
    return '', history

with gr.Blocks(title='Nestle HR Assistant') as demo:
    gr.Markdown('# Nestle HR Policy Assistant')
    gr.Markdown('Ask questions about the Nestle HR policy document.')

    chatbot = gr.Chatbot(label='Conversation', height=450, type='messages')
    user_input = gr.Textbox(
        label='Your question',
        placeholder='Example: What does the policy say about employee conduct?'
    )
    clear_button = gr.Button('Clear Chat')

    user_input.submit(gradio_chat, inputs=[user_input, chatbot], outputs=[user_input, chatbot])
    clear_button.click(lambda: ('', []), outputs=[user_input, chatbot])

demo.launch()

## 8. Result

This notebook demonstrates the complete workflow required in the assignment:

- setting up the programming environment
- loading and processing the PDF document
- generating embeddings and storing them in Chroma DB
- retrieving relevant chunks for question answering
- using an OpenAI chat model to produce answers
- building a Gradio chatbot interface for interaction